# Advanced Tutorial: Custom Model, Metadata, Properties, and Preprocessing

**Before you start:** Run from the repo (notebook under `Notebooks/tutorials/`). Replace `path/to/images` and path templates with your data. Complete [Getting Started](../../docs/getting-started.md) or the [beginner tutorial](01_beginner_interactive.ipynb) first if you have not run the pipeline yet.

**In plain terms:** The model turns each image into an **embedding** (a numeric summary of appearance). **PCA** plots help you see whether treatments separate in that summary space. **Distance to a reference** (e.g. control) quantifies how “far” each sample is from your chosen baseline—useful for comparing drug or genetic conditions.

This notebook covers path-based metadata, processing options, optional distances and plots, property presets, and (optionally) swapping the vision model via a custom wrapper. See [Custom Metadata](../../docs/guides/custom-metadata.md), [Custom Properties](../../docs/guides/custom-properties.md), [Best Practices](../../docs/guides/best-practices.md), and [Common Workflows](../../docs/examples/workflows.md). Custom models: [Vision Model Wrappers](../../docs/reference/api/model-wrapper.md).

**Author:** Aitor González-Marfil ([@AAitorG](https://github.com/AAitorG))

---

## 1. Setup and Reproducibility

Use `PhenoMe(seed=42)` for reproducibility. For full experiment tracking, use `export_experiment_config()` after `save_results()`.

In [ ]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)


from phenome import PhenoMe, get_metadata_from_path, load_dinov2_model

## 2. Path-Based Metadata Extraction

For folder structure `.../(condition)/(timepoint)/(filename).*` (e.g., `data/Treated/24h/cell_001.tif`), use `get_metadata_from_path`.

In [ ]:
pheno = PhenoMe(seed=42)

# Template: .../(condition)/(timepoint)/(filename).*
metadata_fn = get_metadata_from_path(".../(condition)/(timepoint)/(filename).*")
file_df = pheno.find_files("path/to/images", metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")
print(f"Metadata keys: {file_df.columns.tolist()}")

## 3. Load Model and Process Images

Use `channel_mode='split'` for fluorescence images (each channel processed independently). Use `'combined'` for brightfield/RGB. See [Channel Modes](../../docs/concepts.md#channel-modes).

In [ ]:
model, wrapper = load_dinov2_model(model_name="dinov2_vitb14_reg")
pheno.process_images(wrapper, channel_mode="split")

# Optional: audit before processing
# audit_df = pheno.audit_data()

## 4. Distance Analysis and Visualization (Optional)

Compute distances to a reference group (e.g. all samples vs `condition='Control'`). **Reading the plots:** points that cluster together are similar in embedding space; the distance distribution shows how far each group lies from the reference centroid.

Use `get_embeddings()` for lazy-safe access when using checkpoints (not shown here). See [Distances](../../docs/reference/api/distances.md) and [Visualization](../../docs/reference/api/visualization.md).

In [ ]:
# Compute distances to reference group
dist_results = pheno.compute_reference_distances(
    reference_filters={"condition": "Control"}, source="embeddings", mode="centroid"
)

# Visualize
pheno.plot_pca(color_by="condition")
pheno.plot_distance_distribution(distance_results=dist_results, group_by="condition", kde=True)

## 5. Property Presets

Use `property_preset='basic'` for shape + intensity. See [Property Reference](../../docs/guides/property-reference.md) for preset contents. Presets: `basic`, `regionprops`, `intensity`, `full`, `full_extended`.

In [ ]:
# Requires masks (mask_dir in find_files) for shape-based presets
df_props = pheno.compute_properties(property_preset="basic")
print(f"Computed {len(df_props.columns)} properties")

## 6. Custom Model Wrapper (Optional)

Only needed if you want a backbone other than DINOv2 (or the other built-in loaders). Subclass `ModelWrapper` and implement `_get_embeddings(tensor)` to return `(B, D)`. See [Vision Model Wrappers](../../docs/reference/api/model-wrapper.md).

In [ ]:
# Example: ResNet model wrapper
# from torchvision.models import resnet50, ResNet50_Weights
#
# class ResNetModelWrapper(ModelWrapper):
#     def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
#         out = self.model(tensor)
#         if out.ndim == 4:
#             out = out.flatten(1)
#         return out
#
# model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
# model = torch.nn.Sequential(*list(model.children())[:-1])
# wrapper = ResNetModelWrapper(model)

## 7. Save Results and Export Config

For reproducibility, save the checkpoint and export experiment config (reference_filters, model_name, etc.) not stored in the checkpoint.

In [ ]:
pheno.save_results(output_dir="results/")

# Optional: save extra params not in checkpoint
pheno.export_experiment_config(
    "results/config.json",
    reference_filters={"condition": "Control"},
    model_name="dinov2_vitb14_reg",
)

## Next Steps

- [04_exploratory_analysis.ipynb](../examples/04_exploratory_analysis.ipynb): After embeddings—`compute_clustering`, `find_prototypes`, `detect_outliers`, grouped property stats, two correlation styles
- [03_data_auditing_tutorial.ipynb](03_data_auditing_tutorial.ipynb): Data validation before processing
- [example_advanced_phenotyping.py](../examples/example_advanced_phenotyping.py): Full script-style pipeline with the same analysis building blocks
- [Common Workflows](../../docs/examples/workflows.md): Drug screening, time-course, property-based analysis
- [Extending the Pipeline](../../docs/guides/extending.md): Extension points